# LogRAG on Kaggle: shared-protocol BGL

This notebook configures BGL LogRAG against `shared-benchmark-v1` using Ollama with the Mistral-7B-Instruct-v0.1 Q4_K_M GGUF from Hugging Face and local Hugging Face embeddings. It does not need an API key. Enable Kaggle Internet and a GPU accelerator, then attach the `dataset-structured` input containing `bgl_structured.jsonl` and the `benchmark_protocol_v1` input. The notebook converts the parsed JSONL into the required CSV, then downloads the BigLog encoder checkpoint into `/kaggle/working/BigLog/pretrained`. The original `mistralai/Mistral-7B-Instruct-v0.1` repository is Safetensors; Ollama needs the GGUF conversion configured below. HDFS remains separate. The benchmark cell is disabled until you explicitly set `RUN_FULL_BENCHMARK = True`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('/kaggle/working/code-papers-doing-copilot')
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/rubyhcm/code-papers-doing-copilot.git',
        str(REPO_DIR),
    ], check=True)
print('Repository:', REPO_DIR)

In [ ]:
# Retain Kaggle's preinstalled CUDA-enabled torch/torchvision.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-r',
    str(REPO_DIR / 'LogRAG' / 'requirements-kaggle.txt'),
], check=True)

In [ ]:
import csv
import json
import shutil
from pathlib import Path

KAGGLE_INPUT_DIR = Path('/kaggle/input')
structured_matches = sorted(KAGGLE_INPUT_DIR.rglob('bgl_structured.jsonl'))
if len(structured_matches) != 1:
    raise FileNotFoundError(
        'Expected exactly one bgl_structured.jsonl under /kaggle/input; '
        f'found {structured_matches}'
    )
BGL_STRUCTURED_JSONL = structured_matches[0]

protocol_matches = sorted(
    path.parent
    for path in KAGGLE_INPUT_DIR.rglob('bgl.csv')
    if (path.parent / 'manifest.json').is_file()
)
if len(protocol_matches) != 1:
    raise FileNotFoundError(
        'Expected exactly one protocol directory containing bgl.csv and '
        f'manifest.json; found {protocol_matches}'
    )
PROTOCOL_DIR = protocol_matches[0]
protocol_manifest = json.loads(
    (PROTOCOL_DIR / 'manifest.json').read_text(encoding='utf-8')
)
if protocol_manifest.get('protocol_version') != 'shared-benchmark-v1':
    raise ValueError(f'Unexpected protocol manifest: {PROTOCOL_DIR / "manifest.json"}')

with (PROTOCOL_DIR / 'bgl.csv').open(encoding='utf-8') as protocol_file:
    protocol_rows = list(csv.DictReader(protocol_file))
if not protocol_rows:
    raise ValueError(f'BGL protocol is empty: {PROTOCOL_DIR / "bgl.csv"}')
window_size = int(protocol_rows[0]['end']) - int(protocol_rows[0]['start'])
expected_records = (
    len(protocol_rows) * window_size
    + int(protocol_manifest['bgl_dropped_incomplete_records'])
)

BGL_STRUCTURED_CSV = Path('/kaggle/working/lograg-inputs/bgl_structured.csv')
BGL_STRUCTURED_CSV.parent.mkdir(parents=True, exist_ok=True)
temporary_csv = BGL_STRUCTURED_CSV.with_suffix('.csv.tmp')
record_count = 0
with BGL_STRUCTURED_JSONL.open(encoding='utf-8') as source, temporary_csv.open(
    'w', newline='', encoding='utf-8'
) as destination:
    writer = csv.writer(destination)
    writer.writerow(['LineId', 'Label', 'Timestamp', 'Content', 'EventTemplate'])
    for record_count, line in enumerate(source, start=1):
        record = json.loads(line)
        line_id = int(record['line_id'])
        if line_id != record_count:
            raise ValueError(f'Non-contiguous BGL line_id at record {record_count}: {line_id}')
        raw_fields = record['raw'].split(maxsplit=9)
        if len(raw_fields) < 9:
            raise ValueError(f'Cannot parse BGL raw record {line_id}: {record["raw"]!r}')
        content = raw_fields[9] if len(raw_fields) == 10 else ''
        writer.writerow([
            line_id,
            record['label'],
            raw_fields[1],
            content,
            record['template'],
        ])

if record_count != expected_records:
    raise ValueError(
        f'BGL JSONL has {record_count:,} records; protocol expects '
        f'{expected_records:,} including the incomplete tail.'
    )
temporary_csv.replace(BGL_STRUCTURED_CSV)
print('BGL JSONL:', BGL_STRUCTURED_JSONL)
print('Protocol:', PROTOCOL_DIR)
print('Converted CSV:', BGL_STRUCTURED_CSV)
print('Records:', f'{record_count:,}', '| free working space (GiB):', round(shutil.disk_usage('/kaggle/working').free / (1024 ** 3), 1))

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec('gdown') is None:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'gdown==5.2.0'],
        check=True,
    )
import gdown

BIGLOG_REPO_DIR = Path('/kaggle/working/BigLog')
if not BIGLOG_REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/BiglogOpenSource/PretrainedModel.git',
        str(BIGLOG_REPO_DIR),
    ], check=True)

BIGLOG_ENCODER = BIGLOG_REPO_DIR / 'pretrained'
if not (BIGLOG_ENCODER / 'config.json').is_file():
    raise FileNotFoundError(f'BigLog config not found in {BIGLOG_ENCODER}')

BIGLOG_CHECKPOINT = BIGLOG_ENCODER / 'pytorch_model.bin'
if not BIGLOG_CHECKPOINT.is_file() or BIGLOG_CHECKPOINT.stat().st_size < 1_000_000:
    downloaded = gdown.download(
        'https://drive.google.com/file/d/12phqzt2QiAHjaO-BUDSyzNcmdg835rJG/view?usp=sharing',
        str(BIGLOG_CHECKPOINT),
        quiet=False,
        fuzzy=True,
    )
    if not downloaded:
        raise RuntimeError('BigLog checkpoint download failed; check Kaggle Internet access.')

if BIGLOG_CHECKPOINT.stat().st_size < 1_000_000:
    raise RuntimeError(f'BigLog checkpoint is missing or incomplete: {BIGLOG_CHECKPOINT}')
print('BigLog encoder ready:', BIGLOG_ENCODER)
print('Checkpoint size (MiB):', round(BIGLOG_CHECKPOINT.stat().st_size / (1024 ** 2), 1))

In [ ]:
import os
import shutil
import subprocess
import time
import requests

OLLAMA_MODEL = 'hf.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF:Q4_K_M'
OLLAMA_BASE_URL = 'http://127.0.0.1:11434'
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
os.environ['OLLAMA_BASE_URL'] = OLLAMA_BASE_URL

if shutil.which('ollama') is None:
    if shutil.which('zstd') is None:
        if shutil.which('apt-get') is None:
            raise RuntimeError('Ollama installer needs zstd; apt-get is unavailable.')
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', 'zstd'], check=True)
    subprocess.run(
        ['bash', '-lc', 'curl -fsSL https://ollama.com/install.sh | sh'],
        check=True,
    )

OLLAMA_LOG = open('/kaggle/working/ollama.log', 'w', encoding='utf-8')
OLLAMA_PROCESS = None
try:
    requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=2).raise_for_status()
except requests.RequestException:
    OLLAMA_PROCESS = subprocess.Popen(
        ['ollama', 'serve'],
        env=os.environ.copy(),
        stdout=OLLAMA_LOG,
        stderr=subprocess.STDOUT,
    )
    for attempt in range(60):
        if OLLAMA_PROCESS.poll() is not None:
            raise RuntimeError('Ollama server exited; inspect /kaggle/working/ollama.log')
        try:
            requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=2).raise_for_status()
            break
        except requests.RequestException:
            if attempt == 59:
                raise TimeoutError('Ollama server did not become ready; inspect ollama.log')
            time.sleep(2)

subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
smoke_response = requests.post(
    f'{OLLAMA_BASE_URL}/api/chat',
    json={
        'model': OLLAMA_MODEL,
        'messages': [{'role': 'user', 'content': 'Reply with READY in one short sentence.'}],
        'stream': False,
        'keep_alive': 0,
        'options': {'temperature': 0, 'num_predict': 64},
    },
    timeout=300,
)
if not smoke_response.ok:
    raise RuntimeError(
        f'Ollama smoke test failed ({smoke_response.status_code}): '
        f'{smoke_response.text}'
    )
smoke_content = smoke_response.json().get('message', {}).get('content', '').strip()
if not smoke_content:
    raise RuntimeError(f'Ollama returned an empty response: {smoke_response.text}')
print('Ollama model ready:', OLLAMA_MODEL)
print('Smoke response:', smoke_content[:200])

In [ ]:
import os
from datetime import datetime
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
assert BGL_STRUCTURED_CSV.is_file(), BGL_STRUCTURED_CSV
assert (BIGLOG_ENCODER / 'config.json').is_file(), BIGLOG_ENCODER
assert (PROTOCOL_DIR / 'bgl.csv').is_file(), PROTOCOL_DIR
assert (PROTOCOL_DIR / 'manifest.json').is_file(), PROTOCOL_DIR
print('BGL CSV:', BGL_STRUCTURED_CSV)
print('Protocol directory:', PROTOCOL_DIR)

os.environ['LOGRAG_CHAT_MODEL'] = OLLAMA_MODEL
os.environ['LOGRAG_LLM_BACKEND'] = 'ollama'
os.environ['LOGRAG_EMBEDDING_BACKEND'] = 'huggingface'
os.environ['LOGRAG_EMBEDDING_MODEL'] = 'sentence-transformers/all-MiniLM-L6-v2'
os.environ['LOGRAG_EMBEDDING_DEVICE'] = 'cpu'
os.environ['LOGRAG_OLLAMA_NUM_CTX'] = '4096'
os.environ['LOGRAG_OLLAMA_NUM_PREDICT'] = '512'
os.environ['LOGRAG_STRUCTURED_CSV'] = str(BGL_STRUCTURED_CSV)
os.environ['LOGRAG_ENCODER_PATH'] = str(BIGLOG_ENCODER)
os.environ['LOGRAG_PROTOCOL_DIR'] = str(PROTOCOL_DIR)
RUN_ID = datetime.now().strftime('%Y%m%d-%H%M%S')
os.environ['LOGRAG_SHARED_OUTPUT_DIR'] = f'/kaggle/working/lograg-results-{RUN_ID}'
os.environ['LOGRAG_OUTPUT_DIR'] = os.environ['LOGRAG_SHARED_OUTPUT_DIR']
os.environ['LOGRAG_PERSIST_DIRECTORY'] = f'/kaggle/working/lograg-results-{RUN_ID}/ragdb-bgl'
os.environ['LOGRAG_SEED'] = '42'
os.environ['LOGRAG_EPOCHS'] = '50'
os.environ['LOGRAG_BATCH_SIZE'] = '40960'
os.environ['LOGRAG_EMBEDDING_BATCH_SIZE'] = '32'

print('GPU:', torch.cuda.get_device_name(0))
print('Local Ollama chat and Hugging Face embeddings configured; no API key is used.')

In [ ]:
import os
os.chdir(REPO_DIR / 'LogRAG')
sys.path.insert(0, str(REPO_DIR / 'LogRAG'))
from shared_benchmark import prepare_bgl_splits

SPLIT_DIR = Path(os.environ['LOGRAG_SHARED_OUTPUT_DIR']) / 'input_splits'
split_paths = prepare_bgl_splits(BGL_STRUCTURED_CSV, PROTOCOL_DIR, SPLIT_DIR)
for split, path in split_paths.items():
    with open(path, encoding='utf-8') as stream:
        record_count = sum(1 for _ in stream) - 1
    print(split, 'records:', record_count)

The preflight above checks row order, required columns, exact 200-record windows, labels, and record counts against the manifest. Ollama uses the Q4_K_M GGUF; the separate sentence-transformers embedding model is downloaded from Hugging Face and runs on CPU. Review available GPU/disk space before enabling the full training cell. This setup uses the plan's 50-epoch profile; document that configuration in the thesis.

In [ ]:
RUN_FULL_BENCHMARK = False
if RUN_FULL_BENCHMARK:
    subprocess.run([sys.executable, 'main_shared.py'], check=True)
else:
    print('Preflight passed. Set RUN_FULL_BENCHMARK=True to train and run local Ollama RAG.')

In [ ]:
RESULTS_DIR = Path(os.environ['LOGRAG_SHARED_OUTPUT_DIR'])
if (RESULTS_DIR / 'results.json').exists():
    print((RESULTS_DIR / 'results.json').read_text(encoding='utf-8'))
    print('Download the lograg-results output directory from the Kaggle notebook session.')
else:
    print('No benchmark results yet; run the full benchmark cell first.')